# Perbandingan Metode Peramalan Beban Listrik per Jam Panama
### *Time Series Foundation Model* vs Metode Statistik, *Machine Learning*, dan *Deep Learning*

---

## Daftar isi
1. Pendahuluan
2. Persiapan (instalasi, pengaturan, data)
3. Eksplorasi data
4. Metodologi (skema uji, model, metrik)
5. Fungsi model
6. Menjalankan semua model
7. Hasil evaluasi utama
8. *Rolling-origin cross-validation*
9. Pembahasan, keterbatasan, dan kesimpulan
10. Simpan semua hasil ke CSV

# 1. Pendahuluan

## 1.1 Latar belakang

Peramalan beban listrik jangka pendek dipakai operator kelistrikan untuk menjadwalkan pembangkit. Ramalan yang terlalu rendah berisiko menimbulkan pemadaman, sedangkan ramalan yang terlalu tinggi membuat biaya operasi membengkak.

Saat ini muncul pendekatan baru, yaitu ***Time Series Foundation Model* (TSFM)**: model berukuran besar yang **dilatih lebih dulu pada jutaan deret waktu dari berbagai bidang**, lalu bisa langsung meramal data baru tanpa dilatih ulang. Kemampuan ini disebut ***zero-shot forecasting***.

Pertanyaan utama penelitian ini: **apakah model fondasi benar-benar lebih akurat daripada metode yang sudah mapan?** Untuk menjawabnya, tiga model fondasi diadu dengan pembanding dari berbagai kelompok metode, menggunakan skema uji dan metrik yang sama.

## 1.2 Tujuan

1. Membandingkan akurasi **tiga model fondasi** (TimeGPT, Chronos-2, TimesFM 2.5) dengan metode **patokan, statistik, *machine learning*, dan *deep learning*** dalam meramal beban listrik **168 jam (7 hari)** ke depan.
2. Menilai manfaat **variabel eksogen** (suhu dan kalender).
3. Menilai kualitas **selang prediksi** dan **signifikansi** perbedaan akurasi.
4. Menguji **kestabilan** hasil melalui *rolling-origin cross-validation*.

## 1.3 Model fondasi yang diuji

| No | Model fondasi | Pengembang | Akses |
|----|---------------|------------|-------|
| 1 | TimeGPT | Nixtla | API (tertutup) |
| 2 | Chronos-2 | Amazon | *Open-source*, dijalankan lokal |
| 3 | TimesFM 2.5 | Google | *Open-source*, dijalankan lokal |

Semua model, termasuk pembanding, dijalankan **di notebook ini** dengan data, periode uji, dan rumus metrik yang sama.

# 2. Persiapan

## 2.1 Instalasi paket

Jalankan sel di bawah **satu kali saja**. Jika Colab meminta *restart session* setelah instalasi, klik **Restart**, lalu jalankan **Runtime → Run all** lagi (sel instalasi akan selesai cepat karena paket sudah terpasang). Disarankan memakai **Google Colab** (*Runtime → Change runtime type → T4 GPU*), karena Chronos-2, TimesFM, dan N-HiTS memakai PyTorch yang jauh lebih cepat dengan GPU. Semua model tetap bisa berjalan di CPU, hanya lebih lambat.

| Paket | Kegunaan |
|-------|----------|
| `statsforecast` | MSTL, TBATS, ARIMA (DHR) |
| `lightgbm` | LightGBM |
| `neuralforecast` | N-HiTS (*deep learning*) |
| `nixtla` | TimeGPT (API) |
| `chronos-forecasting` | Chronos-2 |
| `timesfm[torch]` | TimesFM 2.5 |
| `scikit-learn` | Dibutuhkan LightGBM dan TimesFM (kovariat). TimesFM juga memakai `jax`, yang sudah tersedia di Colab |

In [ ]:
import subprocess, sys, importlib

# Dipasang satu per satu: bila satu paket gagal, paket lain tetap terpasang
PAKET = [("statsforecast", "statsforecast"), ("lightgbm", "lightgbm"), ("scikit-learn", "sklearn"),
         ("openpyxl", "openpyxl"), ("nixtla", "nixtla"), ("neuralforecast", "neuralforecast"),
         ("chronos-forecasting>=2.2", "chronos"), ("timesfm[torch]", "timesfm")]

for pip_nama, modul in PAKET:
    if importlib.util.find_spec(modul) is not None:
        print(f"{pip_nama:<26} sudah terpasang")
        continue
    hasil = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_nama],
                           capture_output=True, text=True)
    print(f"{pip_nama:<26} " + ("berhasil dipasang" if hasil.returncode == 0
                                 else "GAGAL dipasang:\n" + hasil.stderr[-500:]))
importlib.invalidate_caches()

Sel berikut memeriksa apakah setiap paket benar-benar bisa dipakai. Paket **wajib** adalah `statsforecast` dan `lightgbm`. Bila paket model fondasi atau `neuralforecast` tidak tersedia, model terkait akan dilewati otomatis.

In [ ]:
for pip_nama, modul in PAKET:
    try:
        importlib.import_module(modul)
        print(f"OK     {modul}")
    except Exception as e:
        print(f"TIDAK  {modul}  ->  {type(e).__name__}: {str(e)[:150]}")

## 2.2 Pengaturan

Bagian ini satu-satunya yang perlu disesuaikan:

* **`MY_API_KEY`**: API key TimeGPT dari <https://dashboard.nixtla.io> (menu *API Keys*). Akun gratis mendapat 10.000 panggilan per bulan, sedangkan notebook ini hanya memakai sekitar 8 panggilan.
* **`FILE_XLSX`**: lokasi file data. Di Colab, unggah file lewat panel *Files* di sebelah kiri.
* **Saklar `JALANKAN_...`**: ubah ke `False` untuk melewati model tertentu, misalnya bila belum punya API key atau komputer terlalu lambat.

**Cache.** Hasil model yang berat (model fondasi dan N-HiTS) disimpan di folder `cache_model/`. Saat notebook dijalankan ulang, hasil diambil dari cache sehingga lebih cepat dan kuota API tidak terpakai lagi. **Hapus folder ini** bila pengaturan di bawah diubah.

In [ ]:
MY_API_KEY = "TEMPEL_KEY_DI_SINI"                      # API key TimeGPT
FILE_XLSX  = "Panama_beban_listrik_per_jam_2015-2020_.xlsx"

JALANKAN_TIMEGPT  = True    # Model fondasi 1
JALANKAN_CHRONOS  = True    # Model fondasi 2
JALANKAN_TIMESFM  = True    # Model fondasi 3
JALANKAN_LIGHTGBM = True
JALANKAN_TBATS    = True
JALANKAN_NHITS    = True
JALANKAN_CV       = True    # rolling-origin cross-validation
N_ORIGIN          = 4       # jumlah minggu uji pada cross-validation

H         = 168             # horizon: 7 hari x 24 jam
N_KONTEKS = 24 * 365        # panjang data konteks/latih: 1 tahun
EKSOGEN   = ["T2M_toc", "T2M_san", "T2M_dav", "holiday", "school"]
SEED      = 123

DIR_CACHE = "cache_model"

Penjelasan pengaturan:

* **`H = 168`**: semua model meramal 168 jam ke depan.
* **`N_KONTEKS`**: semua model memakai **1 tahun terakhir** sebelum periode uji. Setahun sudah memuat pola harian, mingguan, dan efek pandemi, sementara data lama sebelum pandemi levelnya sudah berbeda.
* **`EKSOGEN`**: variabel eksogen yang sama untuk semua model yang memakainya, yaitu **suhu di tiga kota** serta penanda **hari libur** dan **masa sekolah**. Kelembapan, angin, dan curah hujan tidak dipakai karena pada uji awal justru menambah *noise*.

In [ ]:
import os, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")
os.makedirs(DIR_CACHE, exist_ok=True)
pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.figsize": (12, 4.5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

## 2.3 Data

Data berisi **beban listrik nasional Panama (MWh) per jam** beserta suhu di Tocumen (Kota Panama), Santiago, dan David, serta penanda hari libur dan masa sekolah. Waktu tercatat dalam zona waktu Panama (UTC−05:00).

In [ ]:
raw  = pd.read_excel(FILE_XLSX, sheet_name="data")
desk = pd.read_excel(FILE_XLSX, sheet_name="deskripsi")

dat = (raw.rename(columns={"datetime": "ds", "nat_demand": "y"})
          [["ds", "y"] + EKSOGEN]
          .sort_values("ds").reset_index(drop=True))
dat["ds"] = pd.to_datetime(dat["ds"])
n = len(dat)

lengkap = pd.date_range(dat["ds"].min(), dat["ds"].max(), freq="h")
print(f"Jumlah observasi : {n:,}".replace(",", "."))
print(f"Periode          : {dat['ds'].min():%d %b %Y %H:%M} s.d. {dat['ds'].max():%d %b %Y %H:%M}")
print(f"Jam yang hilang  : {len(lengkap) - n}")
print(f"Nilai kosong     : {raw.isna().sum().sum()}")
desk

Data tersusun rapi: tidak ada jam yang terlewat dan tidak ada nilai kosong, jadi bisa langsung dimodelkan tanpa imputasi.

In [ ]:
dat["y"].describe().to_frame("Beban (MWh)").T

# 3. Eksplorasi Data

Sebelum memodelkan, kita perlu memahami pola dalam data. Pola inilah yang nanti harus ditangkap oleh setiap model.

## 3.1 Tren dan efek pandemi

In [ ]:
bulanan = dat.set_index("ds")["y"].resample("MS").mean()
feb20, apr20 = bulanan["2020-02-01"], bulanan["2020-04-01"]

fig, ax = plt.subplots()
ax.plot(bulanan.index, bulanan.values, color="steelblue", lw=2)
ax.axvline(pd.Timestamp("2020-03-01"), color="red", ls="--")
ax.text(pd.Timestamp("2020-02-10"), bulanan.max(), "Pandemi COVID-19", color="red", ha="right")
ax.set(title="Rata-rata beban listrik bulanan", ylabel="Beban (MWh)")
plt.show()

print(f"Februari 2020: {feb20:.0f} MWh | April 2020: {apr20:.0f} MWh | turun {(feb20-apr20)/feb20*100:.1f}%")

**Interpretasi:**
* Dari 2015 sampai 2019 beban listrik **naik perlahan**, sejalan dengan pertumbuhan ekonomi dan populasi.
* Ada **pola musiman tahunan**: beban cenderung paling tinggi sekitar **Maret–Juni**, bertepatan dengan akhir musim kemarau saat penggunaan pendingin ruangan meningkat.
* Mulai **Maret 2020** beban **turun tajam** akibat pembatasan aktivitas selama pandemi. Bagi model, ini adalah **perubahan struktural** yang harus bisa diikuti.

## 3.2 Pola harian dan mingguan

In [ ]:
hari_id = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]
pra = dat[dat["ds"] < "2020-03-01"].copy()
pra["jam"], pra["hari"] = pra["ds"].dt.hour, pra["ds"].dt.dayofweek
profil = pra.groupby(["hari", "jam"])["y"].mean().unstack(0)

fig, ax = plt.subplots()
for i in range(7):
    ax.plot(profil.index, profil[i], lw=2, label=hari_id[i])
ax.set(title="Profil beban rata-rata per jam menurut hari (2015 - Feb 2020)",
       xlabel="Jam", ylabel="Beban (MWh)", xticks=range(0, 24, 3))
ax.legend(ncol=7, loc="upper center", bbox_to_anchor=(0.5, -0.15), frameon=False)
plt.show()

**Interpretasi:**
* **Pola harian (24 jam):** beban terendah sekitar pukul 03.00–06.00, naik tajam mulai pukul 07.00, memuncak sekitar pukul 11.00–14.00, turun saat jam pulang kerja, lalu muncul **puncak kedua yang lebih kecil sekitar pukul 19.00** saat penggunaan listrik rumah tangga meningkat.
* **Pola mingguan (168 jam):** Sabtu lebih rendah daripada hari kerja, dan Minggu paling rendah.

Jadi data ini memiliki **musiman ganda** (harian dan mingguan). Model yang baik harus mampu menangkap keduanya.

## 3.3 Hubungan dengan suhu

In [ ]:
sampel = dat.sample(5000, random_state=1)
r_suhu = dat["T2M_toc"].corr(dat["y"])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(sampel["T2M_toc"], sampel["y"], s=6, alpha=0.2, color="steelblue")
ax.set(title=f"Suhu di Tocumen vs beban listrik (r = {r_suhu:.2f})",
       xlabel="Suhu 2 m (°C)", ylabel="Beban (MWh)")
plt.show()

Terlihat **hubungan positif yang cukup kuat**: makin panas, makin besar beban listrik, terutama karena pendingin ruangan. Ini dasar dipakainya suhu sebagai **variabel eksogen**.

## 3.4 Nilai ekstrem

In [ ]:
dat.nsmallest(5, "y")[["ds", "y"]]

Ada beberapa nilai yang **jauh di bawah normal**, misalnya pada 20 Januari 2019 siang. Kemungkinan ini pemadaman (*blackout*) atau gangguan pencatatan. Karena kejadiannya jauh sebelum periode uji dan jumlahnya sangat sedikit, nilai ini **dibiarkan apa adanya**.

# 4. Metodologi

## 4.1 Skema pengujian

| Aspek | Ketentuan |
|-------|-----------|
| Periode uji utama | 168 jam terakhir (**origin 1**) |
| Horizon | 168 jam |
| Data konteks/latih | 1 tahun sebelum periode uji (TBATS dan DHR memakai 8 minggu terakhir agar estimasinya tidak terlalu berat) |
| Variabel eksogen | Suhu 3 kota, `holiday`, `school` |
| Validasi tambahan | *Rolling-origin cross-validation*: diulang pada beberapa minggu sebelumnya (origin 2, 3, ...) |

> **Catatan tentang eksogen.** Pada periode uji, suhu yang dipakai adalah **suhu aktual**, setara dengan prakiraan cuaca yang sempurna. Dalam praktik, suhu masa depan berasal dari prakiraan yang mengandung galat, sehingga akurasi model dengan eksogen di sini **sedikit optimistis**. Karena itu, model dengan dan tanpa eksogen **dirangking terpisah**.

In [ ]:
n_origin = N_ORIGIN if JALANKAN_CV else 1

def ambil_split(o):
    # Origin 1 = minggu terakhir; origin 2, 3, ... = minggu-minggu sebelumnya
    akhir = n - (o - 1) * H
    awal  = akhir - H
    hist  = dat.iloc[awal - N_KONTEKS:awal].reset_index(drop=True)
    fut   = dat.iloc[awal:akhir].reset_index(drop=True)
    return hist, fut

pd.DataFrame([{"Origin": o,
               "Awal uji": ambil_split(o)[1]["ds"].iloc[0],
               "Akhir uji": ambil_split(o)[1]["ds"].iloc[-1],
               "Peran": "Evaluasi utama" if o == 1 else "Cross-validation"}
              for o in range(1, n_origin + 1)])

## 4.2 Model yang dibandingkan

| No | Kelompok | Model | Peran | Eksogen | Selang prediksi |
|----|----------|-------|-------|---------|-----------------|
| 1 | Patokan | Seasonal Naive | Batas bawah | Tidak | Tidak |
| 2 | Statistik | MSTL | Klasik untuk musiman ganda | Tidak | Ya |
| 3 | Statistik | TBATS | Pemulusan eksponensial musiman ganda | Tidak | Ya |
| 4 | Statistik | DHR (Fourier + ARIMA) | Klasik dengan eksogen | Ya | Ya |
| 5 | *Machine learning* | LightGBM | Andalan kompetisi beban listrik | Ya | Ya (regresi kuantil) |
| 6 | *Deep learning* | N-HiTS | Jaringan saraf yang dilatih khusus | Ya | Ya |
| 7–8 | Model fondasi | TimeGPT (± eksogen) | Model fondasi (API) | Tidak / Ya | Ya |
| 9–10 | Model fondasi | Chronos-2 (± eksogen) | Model fondasi (open-source) | Tidak / Ya | Ya |
| 11–12 | Model fondasi | TimesFM 2.5 (± eksogen) | Model fondasi (open-source) | Tidak / Ya | Ya (80%) |
| 13 | Ensemble | MSTL + model fondasi terbaik | Model fondasi sebagai pelengkap | Mengikuti | Tidak |

### Penjelasan singkat setiap model

**1. Seasonal Naive.** Ramalan sama dengan nilai pada jam yang sama seminggu sebelumnya, $\hat{y}_t = y_{t-168}$. Model lain harus bisa mengalahkannya agar kerumitannya berarti.

**2. MSTL (*Multiple Seasonal-Trend decomposition using Loess*).** Data dipecah menjadi tren, musiman harian, musiman mingguan, dan sisaan:
$$y_t = T_t + S^{(24)}_t + S^{(168)}_t + R_t$$
Komponen musiman diulang, sedangkan tren dan sisaan diramal dengan ETS.

**3. TBATS.** Model pemulusan eksponensial (*exponential smoothing*) yang merepresentasikan musiman ganda dengan deret trigonometri, ditambah transformasi Box-Cox dan galat ARMA. TBATS mewakili keluarga pemulusan eksponensial musiman ganda, setara dengan *Double Seasonal Holt-Winters* yang belum tersedia di Python.

**4. DHR (*Dynamic Harmonic Regression*).** Musiman ganda diwakili deret Fourier (pasangan sinus–kosinus), ditambah variabel eksogen sebagai regresor, dengan galat ARIMA:
$$y_t = \beta_0 + \sum_{k=1}^{6}\left[a_k \sin\tfrac{2\pi kt}{24} + b_k \cos\tfrac{2\pi kt}{24}\right] + \sum_{k=1}^{4}\left[c_k \sin\tfrac{2\pi kt}{168} + d_k \cos\tfrac{2\pi kt}{168}\right] + \boldsymbol{\gamma}^\top\mathbf{x}_t + \eta_t$$

**5. LightGBM.** Algoritma *gradient boosting* berbasis pohon keputusan. Fitur: jam, hari, suhu, hari libur, masa sekolah, serta **lag 168, 336, dan 504 jam** (beban 1, 2, dan 3 minggu sebelumnya). Lag di bawah 168 jam sengaja tidak dipakai karena nilainya belum tersedia saat meramal 168 jam ke depan. Selang prediksi dibuat dengan **regresi kuantil**.

**6. N-HiTS (*Neural Hierarchical Interpolation for Time Series*).** Jaringan saraf *deep learning* yang dirancang untuk horizon panjang. Model ini memecah ramalan ke beberapa skala waktu (kasar sampai halus). Berbeda dengan model fondasi, N-HiTS **dilatih khusus** pada data ini.

**7–8. TimeGPT (Nixtla).** Model fondasi berbasis *Transformer* yang diakses lewat **API**. Varian `timegpt-1-long-horizon` dipakai karena horizon 168 jam jauh lebih panjang daripada satu siklus musiman (24 jam).

**9–10. Chronos-2 (Amazon).** Model fondasi *open-source* yang dijalankan **di komputer sendiri**. Chronos-2 mendukung ramalan univariat maupun dengan kovariat dalam satu arsitektur.

**11–12. TimesFM 2.5 (Google).** Model fondasi *open-source* berbasis *decoder-only Transformer* dengan 200 juta parameter. Kovariat dimasukkan lewat regresi linier di dalam konteks (*in-context regression*), lalu sisanya diramal TimesFM. TimesFM menghasilkan kuantil 10%–90%, sehingga hanya selang 80% yang tersedia.

**13. Ensemble.** Rata-rata sederhana ramalan MSTL dan model fondasi terbaik, untuk menguji apakah model fondasi berguna sebagai **pelengkap** metode klasik.

## 4.3 Metrik evaluasi

| Kelompok | Metrik | Rumus / definisi | Kriteria baik |
|----------|--------|------------------|---------------|
| Akurasi titik | MAE | $\frac{1}{h}\sum \lvert y_t-\hat y_t\rvert$ | Makin kecil |
| | RMSE | $\sqrt{\frac{1}{h}\sum (y_t-\hat y_t)^2}$ | Makin kecil |
| | MAPE | $\frac{100\%}{h}\sum \left\lvert \frac{y_t-\hat y_t}{y_t}\right\rvert$ | Makin kecil (< 10% sangat baik) |
| | **MASE** | $\dfrac{\text{MAE}}{\frac{1}{T-168}\sum_{t=169}^{T}\lvert y_t-y_{t-168}\rvert}$ | Makin kecil |
| Selang prediksi | Cakupan 80% dan 95% | % nilai aktual di dalam selang | Mendekati 80% dan 95% |
| | Winkler score | Lebar selang $+\frac{2}{\alpha}\times$ jarak aktual di luar selang | Makin kecil |
| Signifikansi | Uji Diebold–Mariano | Uji beda akurasi dua model, dengan koreksi Holm | p < 0,05 = berbeda nyata |
| Praktis | Waktu komputasi | Detik untuk latih + ramal | Makin cepat |

* **MASE dipakai sebagai metrik utama peringkat** karena tidak bergantung pada satuan data. Penyebutnya adalah rata-rata kesalahan Seasonal Naive pada **data konteks**. Jadi MASE = 1 berarti kesalahan model sama dengan kesalahan Seasonal Naive di data latih.
* **Winkler score** dihitung pada selang 95%. Untuk TimesFM yang hanya punya selang 80%, Winkler dihitung pada selang 80% dan diberi catatan.

# 5. Fungsi Model

Setiap model dibuat sebagai **fungsi dengan format yang sama**: menerima data konteks (`hist`) dan data periode yang diramal (`fut`, hanya kolom eksogennya yang dipakai), lalu mengembalikan tabel berisi `ds`, `mean`, `lo80`, `hi80`, `lo95`, `hi95`. Format seragam ini memungkinkan semua model dievaluasi dan diulang pada setiap origin dengan cara yang sama.

## 5.1 Fungsi bantu

In [ ]:
def keluaran(fut, mean, lo80=np.nan, hi80=np.nan, lo95=np.nan, hi95=np.nan):
    # Format keluaran standar semua model
    return pd.DataFrame({"ds": fut["ds"].values,
                         "mean": np.asarray(mean, dtype=float),
                         "lo80": np.broadcast_to(np.asarray(lo80, dtype=float), len(fut)),
                         "hi80": np.broadcast_to(np.asarray(hi80, dtype=float), len(fut)),
                         "lo95": np.broadcast_to(np.asarray(lo95, dtype=float), len(fut)),
                         "hi95": np.broadcast_to(np.asarray(hi95, dtype=float), len(fut))})

def ke_long(d, kolom):
    # Format panjang (unique_id, ds, y, ...) yang dipakai paket Nixtla
    out = d[["ds"] + kolom].copy()
    out.insert(0, "unique_id", "Panama")
    return out

def eksogen_aktif(hist, fut):
    # Buang eksogen yang nilainya konstan pada jendela ini (mis. tidak ada hari libur)
    gab = pd.concat([hist, fut])
    return [k for k in EKSOGEN if gab[k].nunique() > 1]

def fourier(t, periode, K):
    return np.column_stack([f(2 * np.pi * k * t / periode)
                            for k in range(1, K + 1) for f in (np.sin, np.cos)])

## 5.2 Patokan dan model statistik

In [ ]:
# Paket diimpor di dalam fungsi: bila belum terpasang, hanya model terkait yang dilewati
def sf_models():
    from statsforecast import StatsForecast
    from statsforecast.models import MSTL, AutoETS, AutoARIMA, AutoTBATS
    return StatsForecast, MSTL, AutoETS, AutoARIMA, AutoTBATS

def ambil_sf(fc, nama, fut):
    return keluaran(fut, fc[nama], fc[f"{nama}-lo-80"], fc[f"{nama}-hi-80"],
                    fc[f"{nama}-lo-95"], fc[f"{nama}-hi-95"])

# 1. Seasonal Naive
def m_snaive(hist, fut):
    return keluaran(fut, hist["y"].values[-168:][:len(fut)])

# 2. MSTL (+ ETS untuk tren)
def m_mstl(hist, fut):
    StatsForecast, MSTL, AutoETS, _, _ = sf_models()
    sf = StatsForecast(models=[MSTL(season_length=[24, 168], trend_forecaster=AutoETS(model="ZZN"))], freq="h")
    fc = sf.forecast(df=ke_long(hist, ["y"]), h=len(fut), level=[80, 95])
    return ambil_sf(fc, "MSTL", fut)

# 3. TBATS (8 minggu terakhir)
def m_tbats(hist, fut):
    StatsForecast, _, _, _, AutoTBATS = sf_models()
    sf = StatsForecast(models=[AutoTBATS(season_length=[24, 168])], freq="h")
    fc = sf.forecast(df=ke_long(hist.tail(24 * 7 * 8), ["y"]), h=len(fut), level=[80, 95])
    return ambil_sf(fc, "AutoTBATS", fut)

# 4. DHR: Fourier + eksogen + galat ARIMA (8 minggu terakhir)
def m_dhr(hist, fut):
    StatsForecast, _, _, AutoARIMA, _ = sf_models()
    hs  = hist.tail(24 * 7 * 8).reset_index(drop=True)
    eks = eksogen_aktif(hs, fut)
    t   = np.arange(len(hs) + len(fut))
    F   = np.hstack([fourier(t, 24, 6), fourier(t, 168, 4)])
    kol_f = [f"f{i}" for i in range(F.shape[1])]
    dh = ke_long(hs, ["y"] + eks);  dh[kol_f] = F[:len(hs)]
    xf = ke_long(fut, eks);         xf[kol_f] = F[len(hs):]
    sf = StatsForecast(models=[AutoARIMA(season_length=1)], freq="h")
    fc = sf.forecast(df=dh, h=len(fut), X_df=xf, level=[80, 95])
    return ambil_sf(fc, "AutoARIMA", fut)

## 5.3 Machine learning: LightGBM

In [ ]:
KOLOM_LGB = ["jam", "hari"] + EKSOGEN + ["lag168", "lag336", "lag504"]

def fitur_lgb(d):
    d = d.copy()
    d["jam"], d["hari"] = d["ds"].dt.hour, d["ds"].dt.dayofweek
    for L in (168, 336, 504):
        d[f"lag{L}"] = d["y"].shift(L)
    return d

def m_lgbm(hist, fut):
    import lightgbm as lgb
    semua = fitur_lgb(pd.concat([hist, fut.assign(y=np.nan)], ignore_index=True))
    latih = semua.iloc[:len(hist)].dropna(subset=["lag504"])
    uji   = semua.iloc[len(hist):]

    def latih_satu(objective, alpha=None):
        prm = dict(objective=objective, n_estimators=500, learning_rate=0.05, num_leaves=31,
                   min_child_samples=20, colsample_bytree=0.9, random_state=SEED, verbose=-1)
        if alpha is not None:
            prm["alpha"] = alpha
        mdl = lgb.LGBMRegressor(**prm).fit(latih[KOLOM_LGB], latih["y"])
        return mdl.predict(uji[KOLOM_LGB])

    return keluaran(fut, latih_satu("regression"),
                    latih_satu("quantile", 0.10),  latih_satu("quantile", 0.90),
                    latih_satu("quantile", 0.025), latih_satu("quantile", 0.975))

## 5.4 Deep learning: N-HiTS

N-HiTS melihat 2 minggu terakhir (`input_size = 336` jam) untuk meramal 168 jam ke depan, dan memakai variabel eksogen sebagai *future exogenous*. Selang prediksi didapat dari *loss* kuantil (`MQLoss`).

In [ ]:
def m_nhits(hist, fut):
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NHITS
    from neuralforecast.losses.pytorch import MQLoss

    eks = eksogen_aktif(hist, fut)
    model = NHITS(h=len(fut), input_size=2 * 168, futr_exog_list=eks,
                  loss=MQLoss(level=[80, 95]), scaler_type="robust",
                  max_steps=500, random_seed=SEED, enable_progress_bar=False, logger=False)
    nf = NeuralForecast(models=[model], freq="h")
    nf.fit(df=ke_long(hist, ["y"] + eks))
    fc = nf.predict(futr_df=ke_long(fut, eks))
    return keluaran(fut, fc["NHITS-median"], fc["NHITS-lo-80"], fc["NHITS-hi-80"],
                    fc["NHITS-lo-95"], fc["NHITS-hi-95"])

## 5.5 Model fondasi 1: TimeGPT

TimeGPT diakses lewat API dengan paket `nixtla`. Datamu dikirim ke server Nixtla, lalu server mengembalikan ramalan.

In [ ]:
_klien_tgpt = None

def klien_timegpt():
    global _klien_tgpt
    if _klien_tgpt is None:
        from nixtla import NixtlaClient
        key = MY_API_KEY.strip()
        if key in ("", "TEMPEL_KEY_DI_SINI"):
            key = os.environ.get("NIXTLA_API_KEY", "").strip()
        if key in ("", "TEMPEL_KEY_DI_SINI"):
            raise ValueError("API key TimeGPT belum diisi (MY_API_KEY di bagian Pengaturan).")
        _klien_tgpt = NixtlaClient(api_key=key)
        if not _klien_tgpt.validate_api_key():
            raise ValueError("API key ditolak server Nixtla. Buat key baru di dashboard.nixtla.io.")
    return _klien_tgpt

def m_timegpt(hist, fut, pakai_eksogen=False):
    eks = eksogen_aktif(hist, fut) if pakai_eksogen else []
    fc = klien_timegpt().forecast(
        df=ke_long(hist, ["y"] + eks), X_df=ke_long(fut, eks) if eks else None,
        h=len(fut), freq="h", level=[80, 95], model="timegpt-1-long-horizon",
        id_col="unique_id", time_col="ds", target_col="y")
    fc["ds"] = pd.to_datetime(fc["ds"])
    fc = fc.set_index("ds").loc[fut["ds"]]
    return keluaran(fut, fc["TimeGPT"], fc["TimeGPT-lo-80"], fc["TimeGPT-hi-80"],
                    fc["TimeGPT-lo-95"], fc["TimeGPT-hi-95"])

## 5.6 Model fondasi 2: Chronos-2

Chronos-2 diunduh dari Hugging Face (`amazon/chronos-2`) saat pertama kali dipakai, lalu dijalankan di komputer sendiri. Dengan kovariat, kolom eksogen dimasukkan ke data historis (`df`) dan ke data masa depan (`future_df`), sehingga Chronos-2 memperlakukannya sebagai **kovariat yang diketahui di masa depan**.

In [ ]:
_pipa_chronos = None

def pipa_chronos():
    global _pipa_chronos
    if _pipa_chronos is None:
        import torch
        from chronos import BaseChronosPipeline
        perangkat = "cuda" if torch.cuda.is_available() else "cpu"
        _pipa_chronos = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=perangkat)
    return _pipa_chronos

def m_chronos(hist, fut, pakai_eksogen=False):
    eks = eksogen_aktif(hist, fut) if pakai_eksogen else []
    dfh = ke_long(hist, ["y"] + eks)
    dff = ke_long(fut, eks) if eks else None
    q = [0.025, 0.1, 0.5, 0.9, 0.975]
    fc = pipa_chronos().predict_df(dfh, future_df=dff, prediction_length=len(fut),
                                   quantile_levels=q, id_column="unique_id",
                                   timestamp_column="ds", target="y")
    return keluaran(fut, fc["0.5"], fc["0.1"], fc["0.9"], fc["0.025"], fc["0.975"])

## 5.7 Model fondasi 3: TimesFM 2.5

TimesFM 2.5 diunduh dari Hugging Face (`google/timesfm-2.5-200m-pytorch`). Model perlu "dikompilasi" sekali dengan konfigurasi panjang konteks dan horizon maksimum. Untuk versi dengan kovariat, TimesFM lebih dulu mencocokkan regresi linier antara beban dan kovariat, lalu meramal sisanya (`xreg_mode = "xreg + timesfm"`).

Keluaran kuantil TimesFM berurutan: rata-rata, lalu kuantil 10%, 20%, ..., 90%. Jadi selang 80% diambil dari kuantil 10% dan 90%.

In [ ]:
_model_timesfm = None

def model_timesfm():
    global _model_timesfm
    if _model_timesfm is None:
        import timesfm
        _model_timesfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
        _model_timesfm.compile(timesfm.ForecastConfig(
            max_context=8192, max_horizon=256, normalize_inputs=True,
            use_continuous_quantile_head=True, force_flip_invariance=True,
            infer_is_positive=True, fix_quantile_crossing=True,
            return_backcast=True))          # wajib True agar bisa memakai kovariat
    return _model_timesfm

def m_timesfm(hist, fut, pakai_eksogen=False):
    mdl, h = model_timesfm(), len(fut)
    y = hist["y"].values.astype(float)
    if not pakai_eksogen:
        titik, kuantil = mdl.forecast(horizon=h, inputs=[y])
    else:
        eks = eksogen_aktif(hist, fut)
        gab = pd.concat([hist, fut], ignore_index=True)
        num = {k: [gab[k].values.astype(float)] for k in eks if k.startswith("T2M")}
        kat = {k: [gab[k].values.astype(int)] for k in eks if not k.startswith("T2M")}
        titik, kuantil = mdl.forecast_with_covariates(
            inputs=[y], dynamic_numerical_covariates=num or None,
            dynamic_categorical_covariates=kat or None, xreg_mode="xreg + timesfm")
    # Keluaran bisa berisi backcast (karena return_backcast=True) + ramalan.
    # Ramalan selalu berada di h nilai TERAKHIR.
    titik   = np.asarray(titik[0])[-h:]
    kuantil = np.asarray(kuantil[0])[-h:]
    return keluaran(fut, titik, kuantil[:, 1], kuantil[:, 9])

## 5.8 Daftar model

Semua model didaftarkan di sini beserta kelompoknya, apakah memakai eksogen, dan apakah termasuk model fondasi. Model yang berat (`cache=True`) hasilnya disimpan agar tidak perlu dijalankan ulang.

In [ ]:
daftar = [
    dict(nama="Seasonal Naive", kelompok="Patokan",   eks=False, fondasi=False, cache=False, f=m_snaive),
    dict(nama="MSTL",           kelompok="Statistik", eks=False, fondasi=False, cache=False, f=m_mstl),
]
if JALANKAN_TBATS:
    daftar.append(dict(nama="TBATS", kelompok="Statistik", eks=False, fondasi=False, cache=True, f=m_tbats))
daftar.append(dict(nama="DHR + eksogen", kelompok="Statistik", eks=True, fondasi=False, cache=False, f=m_dhr))
if JALANKAN_LIGHTGBM:
    daftar.append(dict(nama="LightGBM + eksogen", kelompok="Machine learning", eks=True, fondasi=False, cache=False, f=m_lgbm))
if JALANKAN_NHITS:
    daftar.append(dict(nama="N-HiTS + eksogen", kelompok="Deep learning", eks=True, fondasi=False, cache=True, f=m_nhits))
if JALANKAN_TIMEGPT:
    daftar += [dict(nama="TimeGPT",           kelompok="Model fondasi", eks=False, fondasi=True, cache=True,
                    f=lambda h_, f_: m_timegpt(h_, f_, False)),
               dict(nama="TimeGPT + eksogen", kelompok="Model fondasi", eks=True,  fondasi=True, cache=True,
                    f=lambda h_, f_: m_timegpt(h_, f_, True))]
if JALANKAN_CHRONOS:
    daftar += [dict(nama="Chronos-2",           kelompok="Model fondasi", eks=False, fondasi=True, cache=True,
                    f=lambda h_, f_: m_chronos(h_, f_, False)),
               dict(nama="Chronos-2 + eksogen", kelompok="Model fondasi", eks=True,  fondasi=True, cache=True,
                    f=lambda h_, f_: m_chronos(h_, f_, True))]
if JALANKAN_TIMESFM:
    daftar += [dict(nama="TimesFM 2.5",           kelompok="Model fondasi", eks=False, fondasi=True, cache=True,
                    f=lambda h_, f_: m_timesfm(h_, f_, False)),
               dict(nama="TimesFM 2.5 + eksogen", kelompok="Model fondasi", eks=True,  fondasi=True, cache=True,
                    f=lambda h_, f_: m_timesfm(h_, f_, True))]

meta = pd.DataFrame([{k: m[k] for k in ("nama", "kelompok", "eks", "fondasi")} for m in daftar]) \
         .rename(columns={"nama": "Model", "kelompok": "Kelompok", "eks": "Eksogen", "fondasi": "Fondasi"})
meta

# 6. Menjalankan Semua Model

Semua model dijalankan pada setiap origin. Waktu komputasi dicatat pada origin 1. Untuk model yang diambil dari cache, waktunya diambil dari saat model pertama kali dijalankan.

> Saat pertama dijalankan, langkah ini bisa memakan waktu cukup lama, terutama TBATS, N-HiTS, Chronos-2, dan TimesFM di CPU. Proses berikutnya jauh lebih cepat karena memakai cache.

> **Bila ada model yang gagal** (misalnya API key salah, model gagal diunduh, atau paket belum terpasang), model itu **dilewati otomatis** dan penyebabnya ditampilkan di akhir sel. Notebook tetap berjalan sampai selesai dengan model lainnya. Setelah masalahnya diperbaiki, jalankan ulang notebook.

In [ ]:
def nama_file(nama, o):
    aman = "".join(c if c.isalnum() else "_" for c in nama)
    return os.path.join(DIR_CACHE, f"{aman}_origin{o}.pkl")

def jalankan(m, hist, fut, o):
    berkas = nama_file(m["nama"], o)
    if m["cache"] and os.path.exists(berkas):
        with open(berkas, "rb") as fh:
            isi = pickle.load(fh)
        return isi["hasil"], isi["waktu"]
    t0 = time.time()
    hasil = m["f"](hist, fut)
    waktu = time.time() - t0
    if m["cache"]:
        with open(berkas, "wb") as fh:
            pickle.dump({"hasil": hasil, "waktu": waktu}, fh)
    return hasil, waktu

ramalan = {}      # ramalan[origin][model] = DataFrame
waktu   = {}
gagal   = {}      # model yang gagal dijalankan beserta pesan errornya

for o in range(1, n_origin + 1):
    hist, fut = ambil_split(o)
    ramalan[o] = {}
    for m in daftar:
        if m["nama"] in gagal:          # sudah gagal di origin sebelumnya, tidak diulang
            continue
        print(f"Origin {o} | {m['nama']:<24}", end=" ")
        try:
            hasil, w = jalankan(m, hist, fut, o)
            if len(hasil) != len(fut) or hasil["mean"].isna().any():
                raise ValueError("ramalan kosong atau panjangnya tidak sesuai horizon")
        except Exception as e:
            gagal[m["nama"]] = f"{type(e).__name__}: {e}"
            print("GAGAL -> model dilewati")
            continue
        ramalan[o][m["nama"]] = hasil
        if o == 1:
            waktu[m["nama"]] = w
        print(f"selesai ({w:.1f} detik)")

# Model yang gagal di salah satu origin dibuang dari semua origin agar perbandingan tetap adil
for o in ramalan:
    for nm in gagal:
        ramalan[o].pop(nm, None)
meta = meta[~meta["Model"].isin(gagal)].reset_index(drop=True)

if gagal:
    print("\nPERHATIAN: model berikut gagal dan TIDAK disertakan dalam evaluasi.")
    print("Notebook tetap berjalan sampai selesai dengan model lainnya.")
    for nm, alasan in gagal.items():
        print(f"  - {nm}: {alasan[:300]}")
else:
    print("\nSemua model berhasil dijalankan.")

## 6.1 Ensemble

Model fondasi terbaik (MASE terkecil pada origin 1) digabung dengan MSTL melalui rata-rata sederhana.

In [ ]:
def skala_mase(o):
    y = ambil_split(o)[0]["y"].values
    return np.mean(np.abs(y[168:] - y[:-168]))

def metrik_titik(y, f, skala):
    e = y - f
    return dict(MAE=np.mean(np.abs(e)), RMSE=np.sqrt(np.mean(e ** 2)),
                MAPE=np.mean(np.abs(e / y)) * 100, MASE=np.mean(np.abs(e)) / skala)

y1 = ambil_split(1)[1]["y"].values
fondasi_ada = [m for m in meta.loc[meta["Fondasi"], "Model"] if m in ramalan[1]]

terbaik_f = None
if fondasi_ada:
    mase_f    = {m: metrik_titik(y1, ramalan[1][m]["mean"].values, skala_mase(1))["MASE"] for m in fondasi_ada}
    terbaik_f = min(mase_f, key=mase_f.get)

if terbaik_f and "MSTL" in ramalan[1]:
    nama_ens  = f"Ensemble (MSTL + {terbaik_f})"
    for o in range(1, n_origin + 1):
        fut = ambil_split(o)[1]
        ramalan[o][nama_ens] = keluaran(fut, (ramalan[o]["MSTL"]["mean"].values +
                                              ramalan[o][terbaik_f]["mean"].values) / 2)
    meta = pd.concat([meta, pd.DataFrame([{"Model": nama_ens, "Kelompok": "Ensemble",
                                           "Eksogen": bool(meta.loc[meta["Model"] == terbaik_f, "Eksogen"].iloc[0]),
                                           "Fondasi": False}])], ignore_index=True)
    print(f"Model fondasi terbaik: {terbaik_f}  ->  dibentuk {nama_ens}")
else:
    print("Tidak ada model fondasi (atau MSTL) yang berhasil dijalankan, ensemble tidak dibentuk.")

# 7. Hasil Evaluasi Utama (Origin 1)

## 7.1 Akurasi titik

In [ ]:
def winkler(y, lo, hi, alpha):
    return np.mean((hi - lo) + (2 / alpha) * np.maximum(lo - y, 0) + (2 / alpha) * np.maximum(y - hi, 0))

def hitung_akurasi(o):
    y, sk = ambil_split(o)[1]["y"].values, skala_mase(o)
    baris = []
    for m, r in ramalan[o].items():
        d = dict(Model=m, origin=o, **metrik_titik(y, r["mean"].values, sk))
        ada80, ada95 = r["lo80"].notna().all(), r["lo95"].notna().all()
        d["Cakupan80"] = np.mean((y >= r["lo80"]) & (y <= r["hi80"])) * 100 if ada80 else np.nan
        d["Cakupan95"] = np.mean((y >= r["lo95"]) & (y <= r["hi95"])) * 100 if ada95 else np.nan
        if ada95:
            d["Winkler"], d["Selang Winkler"] = winkler(y, r["lo95"].values, r["hi95"].values, 0.05), "95%"
        elif ada80:
            d["Winkler"], d["Selang Winkler"] = winkler(y, r["lo80"].values, r["hi80"].values, 0.20), "80%"
        else:
            d["Winkler"], d["Selang Winkler"] = np.nan, "-"
        baris.append(d)
    return pd.DataFrame(baris).merge(meta, on="Model", how="left")

akurasi1 = hitung_akurasi(1)
akurasi1["Waktu (detik)"] = akurasi1["Model"].map(waktu)

def tabel_rank(d):
    return (d.sort_values("MASE").reset_index(drop=True)
             .assign(Peringkat=lambda x: x.index + 1)
             [["Peringkat", "Model", "Kelompok", "MAE", "RMSE", "MAPE", "MASE", "Waktu (detik)"]]
             .rename(columns={"MAPE": "MAPE (%)"}).set_index("Peringkat"))

print("Tabel A. Model TANPA eksogen (diurutkan menurut MASE)")
display(tabel_rank(akurasi1[~akurasi1["Eksogen"]]).round(2))
print("\nTabel B. Model DENGAN eksogen (diurutkan menurut MASE)")
display(tabel_rank(akurasi1[akurasi1["Eksogen"]]).round(2))

Kolom *Waktu* kosong untuk ensemble karena ensemble hanya merata-ratakan dua ramalan yang sudah ada.

In [ ]:
urut      = akurasi1.sort_values("MASE").reset_index(drop=True)
best      = urut.loc[0, "Model"]
best_non  = urut[~urut["Eksogen"]].iloc[0]
best_eks  = urut[urut["Eksogen"]].iloc[0] if urut["Eksogen"].any() else None
fondasi1  = urut[urut["Fondasi"] == True]
klasik1   = urut[(urut["Fondasi"] == False) & (urut["Kelompok"] != "Ensemble")]
mase_sn   = urut.loc[urut["Model"] == "Seasonal Naive", "MASE"].iloc[0]
kalah_sn  = urut.loc[(urut["MASE"] >= mase_sn) & (urut["Model"] != "Seasonal Naive"), "Model"].tolist()
fk = lambda x: f"{x:.2f}".replace(".", ",")

print("RINGKASAN HASIL")
print(f"- Model terbaik secara keseluruhan : {best} (MASE {fk(urut.loc[0,'MASE'])}, MAPE {fk(urut.loc[0,'MAPE'])}%)")
print(f"- Terbaik tanpa eksogen            : {best_non['Model']} (MAPE {fk(best_non['MAPE'])}%)")
if best_eks is not None:
    print(f"- Terbaik dengan eksogen           : {best_eks['Model']} (MAPE {fk(best_eks['MAPE'])}%)")
if len(fondasi1):
    bf, bk = fondasi1.iloc[0], klasik1.iloc[0]
    status = "LEBIH AKURAT daripada" if bf["MASE"] < bk["MASE"] else "BELUM MENGUNGGULI"
    print(f"- Model fondasi terbaik {bf['Model']} (MASE {fk(bf['MASE'])}) {status} "
          f"metode non-fondasi terbaik {bk['Model']} (MASE {fk(bk['MASE'])})")
print(f"- MASE Seasonal Naive pada periode uji: {fk(mase_sn)}")
print("- Model yang tidak mampu mengalahkan Seasonal Naive: " + (", ".join(kalah_sn) if kalah_sn else "tidak ada"))

**Cara membaca:**
* Tabel A dan B dipisah karena model dengan eksogen mendapat informasi suhu aktual (lihat catatan di Bagian 4.1).
* MASE dan MAPE yang kecil berarti akurat. MAPE di bawah 10% umumnya dianggap sangat baik.
* Model yang MASE-nya lebih besar daripada MASE Seasonal Naive berarti **kalah dari patokan paling sederhana**, sehingga kerumitannya tidak memberi manfaat pada minggu uji ini.

In [ ]:
WARNA = {"Patokan": "#999999", "Statistik": "#56B4E9", "Machine learning": "#E69F00",
         "Deep learning": "#009E73", "Model fondasi": "#CC79A7", "Ensemble": "#D55E00"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1, 1]})
for ax, (judul, d) in zip(axes, [("Tanpa eksogen", akurasi1[~akurasi1["Eksogen"]]),
                                 ("Dengan eksogen", akurasi1[akurasi1["Eksogen"]])]):
    d = d.sort_values("MASE", ascending=False)
    ax.barh(d["Model"], d["MASE"], color=d["Kelompok"].map(WARNA))
    ax.axvline(mase_sn, color="black", ls="--", lw=1)
    ax.set(title=judul, xlabel="MASE (makin kecil makin baik)")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in WARNA.values()]
fig.legend(handles, WARNA.keys(), ncol=6, loc="lower center", frameon=False)
fig.suptitle("MASE per model (garis putus-putus = Seasonal Naive pada periode uji)")
plt.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.show()

## 7.2 Visualisasi ramalan

Agar tidak terlalu ramai, ramalan ditampilkan per kelompok metode. Garis hitam adalah beban aktual.

In [ ]:
fut1 = ambil_split(1)[1]
urutan_kelompok = [k for k in WARNA if k in meta["Kelompok"].values]

fig, axes = plt.subplots(len(urutan_kelompok), 1, figsize=(13, 3.2 * len(urutan_kelompok)), sharex=True)
axes = np.atleast_1d(axes)
for ax, kel in zip(axes, urutan_kelompok):
    ax.plot(fut1["ds"], fut1["y"], color="black", lw=2, label="Aktual")
    for m in meta.loc[meta["Kelompok"] == kel, "Model"]:
        if m in ramalan[1]:
            ax.plot(ramalan[1][m]["ds"], ramalan[1][m]["mean"], lw=1.2, label=m)
    ax.set(title=kel, ylabel="MWh")
    ax.legend(fontsize=8, loc="upper left", ncol=3)
fig.suptitle("Ramalan vs aktual pada periode uji utama", y=1.0)
plt.tight_layout()
plt.show()

**Cara membaca:** model yang garisnya paling menempel ke garis hitam adalah yang paling akurat. Perhatikan terutama **puncak siang hari** (apakah model mencapai tinggi puncak yang sebenarnya) dan **akhir pekan** (apakah model menangkap turunnya beban).

## 7.3 Kesalahan menurut jam

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for m, r in ramalan[1].items():
    galat = pd.Series(np.abs(fut1["y"].values - r["mean"].values), index=fut1["ds"].dt.hour)
    ax.plot(galat.groupby(level=0).mean(), lw=1.3, label=m)
ax.set(title="Rata-rata kesalahan absolut menurut jam", xlabel="Jam", ylabel="MAE (MWh)", xticks=range(0, 24, 3))
ax.legend(fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False)
plt.show()

Biasanya kesalahan terbesar terjadi pada **jam-jam puncak**, karena di jam itulah beban paling dipengaruhi suhu dan aktivitas yang berubah-ubah.

## 7.4 Kualitas selang prediksi

In [ ]:
(akurasi1[akurasi1["Cakupan80"].notna()]
   .sort_values(["Selang Winkler", "Winkler"], ascending=[False, True])
   [["Model", "Cakupan80", "Cakupan95", "Winkler", "Selang Winkler"]]
   .rename(columns={"Cakupan80": "Cakupan 80% (%)", "Cakupan95": "Cakupan 95% (%)"})
   .round(2).reset_index(drop=True))

* **Cakupan** yang ideal mendekati nilai nominalnya (80% dan 95%). Cakupan jauh di bawah nominal berarti selang **terlalu sempit** (model terlalu percaya diri). Cakupan mendekati 100% bisa berarti selang **terlalu lebar**.
* **Winkler score** menggabungkan keduanya: selang yang sempit tetapi tetap memuat nilai aktual mendapat skor kecil. Skor hanya bisa dibandingkan antar-model dengan **selang yang sama** (kolom *Selang Winkler*).

In [ ]:
# Contoh selang prediksi: model fondasi terbaik
if terbaik_f:
    r = ramalan[1][terbaik_f]
    hist1 = ambil_split(1)[0].tail(24 * 14)
    fig, ax = plt.subplots(figsize=(13, 4.5))
    ax.plot(hist1["ds"], hist1["y"], color="grey", lw=1, label="Histori")
    if r["lo95"].notna().all():
        ax.fill_between(r["ds"], r["lo95"], r["hi95"], color="#CC79A7", alpha=0.15, label="Selang 95%")
    ax.fill_between(r["ds"], r["lo80"], r["hi80"], color="#CC79A7", alpha=0.3, label="Selang 80%")
    ax.plot(r["ds"], r["mean"], color="#CC79A7", lw=1.5, label=terbaik_f)
    ax.plot(fut1["ds"], fut1["y"], color="black", ls="--", lw=1.2, label="Aktual")
    ax.set(title=f"Histori 2 minggu dan ramalan {terbaik_f} 7 hari", ylabel="Beban (MWh)")
    ax.legend(ncol=5, fontsize=9)
    plt.show()

## 7.5 Uji signifikansi Diebold–Mariano

Uji ini memeriksa apakah selisih akurasi **model terbaik** dengan setiap model lain **nyata secara statistik** atau hanya kebetulan. Galat dibandingkan dengan fungsi kerugian kuadrat. Statistik uji memakai koreksi sampel kecil Harvey–Leybourne–Newbold, dan nilai-p dikoreksi dengan **metode Holm** karena dilakukan banyak uji sekaligus.

In [ ]:
def dm_test(e1, e2, h=1):
    # Uji Diebold-Mariano dengan koreksi Harvey-Leybourne-Newbold (sama seperti forecast::dm.test di R)
    d = e1 ** 2 - e2 ** 2
    T, dbar = len(d), d.mean()
    gamma = [np.sum((d[k:] - dbar) * (d[:T - k] - dbar)) / T for k in range(h)]
    var_d = (gamma[0] + 2 * sum(gamma[1:])) / T
    stat = dbar / np.sqrt(var_d)
    stat *= np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    return stat, 2 * stats.t.cdf(-abs(stat), df=T - 1)

def holm(p):
    p = np.asarray(p); m = len(p); urut = np.argsort(p)
    adj = np.empty(m); maks = 0
    for i, idx in enumerate(urut):
        maks = max(maks, (m - i) * p[idx]); adj[idx] = min(maks, 1)
    return adj

e_best = fut1["y"].values - ramalan[1][best]["mean"].values
baris = []
for m, r in ramalan[1].items():
    if m == best:
        continue
    s, p = dm_test(e_best, fut1["y"].values - r["mean"].values)
    baris.append({"Pembanding": m, "Statistik DM": s, "p-value": p})
dm = pd.DataFrame(baris)
dm["p-value (Holm)"] = holm(dm["p-value"])
dm["Kesimpulan"] = np.where(dm["p-value (Holm)"] < 0.05, "Berbeda nyata", "Tidak berbeda nyata")
dm = dm.sort_values("p-value (Holm)").reset_index(drop=True)

print(f"Uji Diebold-Mariano: {best} vs model lain")
display(dm.style.format({"Statistik DM": "{:.3f}", "p-value": "{:.4g}", "p-value (Holm)": "{:.4g}"}))

n_beda = (dm["Kesimpulan"] == "Berbeda nyata").sum()
print(f"{best} berbeda nyata (lebih akurat) dibandingkan {n_beda} dari {len(dm)} model.")
if n_beda < len(dm):
    print("Terhadap model lainnya keunggulannya belum signifikan, sehingga dapat dianggap setara secara statistik.")

Statistik DM negatif berarti galat model terbaik lebih kecil daripada pembandingnya.

# 8. Rolling-Origin Cross-Validation

Hasil dari satu minggu uji bisa saja kebetulan. Karena itu, semua model diuji ulang pada beberapa minggu berurutan, dan akurasinya dirata-ratakan.

In [ ]:
if JALANKAN_CV and n_origin > 1:
    akurasi_cv = pd.concat([hitung_akurasi(o) for o in range(1, n_origin + 1)], ignore_index=True)
    ringkas_cv = (akurasi_cv.groupby(["Model", "Kelompok", "Eksogen"])
                    .agg(**{"Jumlah origin": ("MASE", "size"), "MASE rata-rata": ("MASE", "mean"),
                            "MASE SD": ("MASE", "std"), "MAPE rata-rata (%)": ("MAPE", "mean")})
                    .reset_index().sort_values("MASE rata-rata").reset_index(drop=True))
    ringkas_cv["Eksogen"] = ringkas_cv["Eksogen"].map({True: "Ya", False: "Tidak"})
    display(ringkas_cv.round(3))

    fig, ax = plt.subplots(figsize=(11, 6))
    urutan = ringkas_cv["Model"].tolist()[::-1]
    for i, m in enumerate(urutan):
        v = akurasi_cv.loc[akurasi_cv["Model"] == m, "MASE"]
        kel = meta.loc[meta["Model"] == m, "Kelompok"].iloc[0]
        ax.scatter(v, [i] * len(v), color=WARNA[kel], alpha=0.6)
        ax.scatter(v.mean(), i, color="black", marker="D", s=50)
    ax.set_yticks(range(len(urutan)), urutan)
    ax.set(title=f"MASE pada {n_origin} minggu uji (belah ketupat hitam = rata-rata)", xlabel="MASE")
    plt.show()

    b = ringkas_cv.loc[0, "Model"]
    print(f"Rata-rata pada {n_origin} minggu uji, model terbaik adalah {b}.")
    print("Hasil ini SAMA dengan evaluasi utama, sehingga konsisten." if b == best else
          f"Hasil ini BERBEDA dengan evaluasi utama ({best}): peringkat model sensitif terhadap minggu yang diuji. "
          "Kesimpulan sebaiknya didasarkan pada rata-rata cross-validation.")
else:
    print("Cross-validation tidak dijalankan (JALANKAN_CV = False).")

Titik-titik yang **menyebar lebar** menunjukkan model yang **tidak stabil**: akurat di satu minggu tetapi buruk di minggu lain. Model yang baik untuk dipakai dalam praktik adalah yang rata-ratanya kecil **dan** sebarannya sempit.

# 9. Pembahasan, Keterbatasan, dan Kesimpulan

## 9.1 Pembahasan

1. **Model fondasi vs metode mapan.** Keunggulan utama model fondasi adalah kepraktisannya: ramalan didapat tanpa memilih model, menentukan orde, atau melatih parameter. Namun, untuk data yang sangat teratur seperti beban listrik, metode klasik yang dirancang untuk musiman ganda (MSTL, TBATS) dan *machine learning* dengan fitur yang tepat (LightGBM) tetap merupakan pesaing yang kuat. Bandingkan Tabel A dan B serta hasil cross-validation untuk melihat posisi ketiga model fondasi.
2. **Perbandingan antar-model fondasi.** TimeGPT bersifat tertutup dan diakses lewat API, sedangkan Chronos-2 dan TimesFM *open-source* dan berjalan di komputer sendiri. Selain akurasi, pertimbangkan juga biaya, privasi data, dan kebutuhan perangkat keras.
3. **Variabel eksogen.** Suhu dan kalender berpotensi membantu menangkap tinggi puncak dan hari-hari tidak biasa. Manfaatnya bergantung pada cara model memakainya: Chronos-2 memakai kovariat langsung di dalam model, TimesFM lewat regresi linier, dan TimeGPT lewat mekanisme internalnya.
4. **Ensemble.** Menggabungkan model fondasi dengan metode klasik sering menurunkan kesalahan, karena kesalahan kedua model tidak selalu searah.
5. **Signifikansi dan kestabilan.** Selisih MAPE yang kecil belum tentu berarti satu model lebih baik. Uji Diebold–Mariano dan cross-validation membantu memastikan kesimpulan tidak hanya kebetulan pada satu minggu uji.

## 9.2 Keterbatasan

1. **Suhu aktual sebagai eksogen** membuat akurasi model dengan eksogen optimistis.
2. **Kemungkinan kebocoran data pada model fondasi.** Dataset beban listrik Panama tersedia publik (Kaggle), sehingga mungkin pernah dipakai melatih sebagian model fondasi. Hal ini perlu dicek pada dokumentasi data pelatihan masing-masing model.
3. **Hiperparameter LightGBM dan N-HiTS tidak dituning** secara khusus. Hasilnya bisa membaik dengan tuning.
4. **Selang prediksi TimesFM hanya 80%**, sehingga Winkler score-nya tidak bisa dibandingkan langsung dengan model yang memakai selang 95%.
5. **Satu deret waktu saja** (Panama). Generalisasi ke data lain perlu diuji.

## 9.3 Kesimpulan

Kesimpulan akhir diisi berdasarkan **Ringkasan Hasil** (Bagian 7.1), **uji Diebold–Mariano** (Bagian 7.5), dan **rata-rata cross-validation** (Bagian 8). Sel berikut merangkumnya secara otomatis.

In [ ]:
kesimpulan = []
kesimpulan.append(f"Pada evaluasi utama, model terbaik secara keseluruhan adalah {best} "
                  f"(MASE {fk(urut.loc[0,'MASE'])}; MAPE {fk(urut.loc[0,'MAPE'])}%).")
if len(fondasi1):
    bf, bk = fondasi1.iloc[0], klasik1.iloc[0]
    kesimpulan.append(f"Model fondasi terbaik adalah {bf['Model']} (MASE {fk(bf['MASE'])}), "
                      f"{'lebih akurat daripada' if bf['MASE'] < bk['MASE'] else 'belum mengungguli'} "
                      f"metode non-fondasi terbaik, {bk['Model']} (MASE {fk(bk['MASE'])}).")
    a = akurasi1.set_index("Model")
    for dasar in ["TimeGPT", "Chronos-2", "TimesFM 2.5"]:
        if dasar in a.index and f"{dasar} + eksogen" in a.index:
            selisih = a.loc[dasar, "MAPE"] - a.loc[f"{dasar} + eksogen", "MAPE"]
            kesimpulan.append(f"Eksogen pada {dasar}: MAPE {'turun' if selisih > 0 else 'naik'} "
                              f"{fk(abs(selisih))} poin persentase.")
if JALANKAN_CV and n_origin > 1:
    kesimpulan.append(f"Rata-rata pada {n_origin} minggu uji, model terbaik adalah {ringkas_cv.loc[0,'Model']} "
                      f"(MASE rata-rata {fk(ringkas_cv.loc[0,'MASE rata-rata'])}).")
if gagal:
    kesimpulan.append("Model yang gagal dijalankan dan tidak dievaluasi: " + ", ".join(gagal) + ".")

print("KESIMPULAN")
for i, k in enumerate(kesimpulan, 1):
    print(f"{i}. {k}")

# 10. Simpan Semua Hasil ke CSV

Semua tabel, data di balik setiap grafik, ramalan, dan kesimpulan disimpan sebagai file CSV di folder **`hasil_csv/`**. Nomor di depan nama file mengikuti urutan bagian di notebook.

| File | Isi | Bagian |
|------|-----|--------|
| `01_ringkasan_data.csv` | Jumlah observasi, periode, jam hilang, nilai kosong | 2.3 |
| `02_deskripsi_variabel.csv` | Daftar dan deskripsi variabel | 2.3 |
| `03_statistik_deskriptif.csv` | Statistik deskriptif beban listrik | 2.3 |
| `04_beban_bulanan.csv` | Rata-rata beban per bulan (data grafik tren) | 3.1 |
| `05_profil_jam_hari.csv` | Rata-rata beban per jam dan hari (data grafik pola) | 3.2 |
| `06_korelasi_eksogen.csv` | Korelasi beban dengan setiap variabel eksogen | 3.3 |
| `07_nilai_ekstrem.csv` | 10 nilai beban terendah | 3.4 |
| `08_periode_uji.csv` | Periode setiap origin | 4.1 |
| `09_daftar_model.csv` | Model, kelompok, eksogen, model fondasi | 5.8 |
| `10_waktu_komputasi.csv` | Waktu latih + ramal setiap model | 6 |
| `11_model_gagal.csv` | Model yang gagal beserta penyebabnya | 6 |
| `12_ramalan_semua_model.csv` | Ramalan per jam semua model di semua origin, beserta aktual dan selang | 6 |
| `13_akurasi_origin1.csv` | Semua metrik semua model pada evaluasi utama | 7.1 |
| `14_peringkat_tanpa_eksogen.csv` | Tabel A | 7.1 |
| `15_peringkat_dengan_eksogen.csv` | Tabel B | 7.1 |
| `16_galat_per_jam.csv` | MAE per jam setiap model (data grafik galat) | 7.3 |
| `17_selang_prediksi.csv` | Cakupan dan Winkler score | 7.4 |
| `18_uji_diebold_mariano.csv` | Hasil uji Diebold–Mariano | 7.5 |
| `19_akurasi_cv_per_origin.csv` | Semua metrik setiap model di setiap origin | 8 |
| `20_ringkasan_cv.csv` | Rata-rata dan simpangan baku MASE dan MAPE | 8 |
| `21_kesimpulan.csv` | Kalimat kesimpulan | 9.3 |

File disimpan dengan *encoding* `utf-8-sig` agar huruf khusus (misalnya °C) tampil benar saat dibuka di Excel. Semua file juga dikemas menjadi **`hasil_csv.zip`** supaya mudah diunduh, misalnya dari Colab.

In [ ]:
import shutil

DIR_HASIL = "hasil_csv"
os.makedirs(DIR_HASIL, exist_ok=True)
tersimpan = []

def simpan(df, nama):
    path = os.path.join(DIR_HASIL, nama)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    tersimpan.append((nama, len(df)))

# --- Data dan eksplorasi ---
simpan(pd.DataFrame({
    "Keterangan": ["Jumlah observasi", "Awal", "Akhir", "Jam yang hilang", "Nilai kosong"],
    "Nilai": [n, dat["ds"].min(), dat["ds"].max(), len(lengkap) - n, int(raw.isna().sum().sum())]}),
    "01_ringkasan_data.csv")
simpan(desk, "02_deskripsi_variabel.csv")
simpan(dat["y"].describe().to_frame("Beban (MWh)").T.reset_index(names="Variabel"), "03_statistik_deskriptif.csv")
simpan(bulanan.reset_index().rename(columns={"ds": "Bulan", "y": "Rata-rata beban (MWh)"}), "04_beban_bulanan.csv")
simpan(profil.rename(columns=dict(enumerate(hari_id))).reset_index().rename(columns={"jam": "Jam"}),
       "05_profil_jam_hari.csv")
simpan(pd.DataFrame({"Variabel": EKSOGEN,
                     "Korelasi dengan beban": [dat[k].corr(dat["y"]) for k in EKSOGEN]}),
       "06_korelasi_eksogen.csv")
simpan(dat.nsmallest(10, "y")[["ds", "y"]].rename(columns={"ds": "Waktu", "y": "Beban (MWh)"}),
       "07_nilai_ekstrem.csv")
simpan(pd.DataFrame([{"Origin": o,
                      "Awal uji": ambil_split(o)[1]["ds"].iloc[0],
                      "Akhir uji": ambil_split(o)[1]["ds"].iloc[-1],
                      "Awal konteks": ambil_split(o)[0]["ds"].iloc[0],
                      "Peran": "Evaluasi utama" if o == 1 else "Cross-validation"}
                     for o in range(1, n_origin + 1)]), "08_periode_uji.csv")

# --- Model dan ramalan ---
simpan(meta, "09_daftar_model.csv")
simpan(pd.DataFrame({"Model": list(waktu), "Waktu (detik)": list(waktu.values())}), "10_waktu_komputasi.csv")
simpan(pd.DataFrame({"Model": list(gagal), "Penyebab": list(gagal.values())}), "11_model_gagal.csv")

eks_map = dict(zip(meta["Model"], meta["Eksogen"]))
ekspor = pd.concat([r.assign(model=m, origin=o, aktual=ambil_split(o)[1]["y"].values,
                             eksogen="ya" if eks_map.get(m) else "tidak")
                    for o in ramalan for m, r in ramalan[o].items()], ignore_index=True)
simpan(ekspor[["origin", "ds", "model", "eksogen", "aktual", "mean", "lo80", "hi80", "lo95", "hi95"]],
       "12_ramalan_semua_model.csv")

# --- Evaluasi utama ---
simpan(akurasi1, "13_akurasi_origin1.csv")
simpan(tabel_rank(akurasi1[~akurasi1["Eksogen"]]).reset_index(), "14_peringkat_tanpa_eksogen.csv")
simpan(tabel_rank(akurasi1[akurasi1["Eksogen"]]).reset_index(), "15_peringkat_dengan_eksogen.csv")

galat_jam = pd.DataFrame({m: pd.Series(np.abs(fut1["y"].values - r["mean"].values),
                                       index=fut1["ds"].dt.hour).groupby(level=0).mean()
                          for m, r in ramalan[1].items()})
simpan(galat_jam.reset_index(names="Jam"), "16_galat_per_jam.csv")

simpan(akurasi1[akurasi1["Cakupan80"].notna()]
         .sort_values(["Selang Winkler", "Winkler"], ascending=[False, True])
         [["Model", "Cakupan80", "Cakupan95", "Winkler", "Selang Winkler"]]
         .rename(columns={"Cakupan80": "Cakupan 80% (%)", "Cakupan95": "Cakupan 95% (%)"}),
       "17_selang_prediksi.csv")
simpan(dm.assign(**{"Model terbaik": best}), "18_uji_diebold_mariano.csv")

# --- Cross-validation ---
if JALANKAN_CV and n_origin > 1:
    simpan(akurasi_cv, "19_akurasi_cv_per_origin.csv")
    simpan(ringkas_cv, "20_ringkasan_cv.csv")

# --- Kesimpulan ---
simpan(pd.DataFrame({"No": range(1, len(kesimpulan) + 1), "Kesimpulan": kesimpulan}), "21_kesimpulan.csv")

shutil.make_archive(DIR_HASIL, "zip", DIR_HASIL)

print(f"{len(tersimpan)} file CSV tersimpan di folder '{DIR_HASIL}/' dan dikemas menjadi '{DIR_HASIL}.zip':")
display(pd.DataFrame(tersimpan, columns=["File", "Jumlah baris"]))

**Mengunduh hasil di Google Colab:** buka panel *Files* (ikon folder di kiri), klik kanan **`hasil_csv.zip`**, lalu pilih *Download*. Atau jalankan sel berikut.

In [ ]:
try:
    from google.colab import files
    files.download("hasil_csv.zip")
except ImportError:
    print("Bukan di Google Colab. File ada di folder kerja:", os.path.abspath("hasil_csv.zip"))

## Referensi

* Ansari, A. F., et al. (2025). Chronos-2: From univariate to universal forecasting. *arXiv:2510.15821*.
* Das, A., Kong, W., Sen, R., & Zhou, Y. (2024). A decoder-only foundation model for time-series forecasting. *ICML 2024*.
* Garza, A., & Mergenthaler-Canseco, M. (2023). TimeGPT-1. *arXiv:2310.03589*.
* Challu, C., et al. (2023). N-HiTS: Neural hierarchical interpolation for time series forecasting. *AAAI 2023*.
* De Livera, A. M., Hyndman, R. J., & Snyder, R. D. (2011). Forecasting time series with complex seasonal patterns using exponential smoothing. *JASA*, 106(496), 1513–1527.
* Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts.
* Hyndman, R. J., & Koehler, A. B. (2006). Another look at measures of forecast accuracy. *International Journal of Forecasting*, 22(4), 679–688.
* Bandara, K., Hyndman, R. J., & Bergmeir, C. (2021). MSTL: A seasonal-trend decomposition algorithm for time series with multiple seasonal patterns. *arXiv:2107.13462*.
* Ke, G., et al. (2017). LightGBM: A highly efficient gradient boosting decision tree. *NeurIPS 30*.
* Diebold, F. X., & Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics*, 13(3), 253–263.
* Harvey, D., Leybourne, S., & Newbold, P. (1997). Testing the equality of prediction mean squared errors. *International Journal of Forecasting*, 13(2), 281–291.
* Winkler, R. L. (1972). A decision-theoretic approach to interval estimation. *JASA*, 67(337), 271–281.